**Note: Run servers on local instead of Colab**

Here are some comments for a tutorial on Model serialization with Pickle/Joblib:

1. **Introduction to Model Serialization**:
   - Model serialization is the process of converting a machine learning model into a format that can be easily saved and loaded. This is crucial for deploying models in production environments or sharing them with others
.

2. **Why Use Pickle or Joblib?**:
   - Pickle and Joblib are two popular libraries in Python for serializing machine learning models. Pickle is a built-in Python module that can serialize and deserialize any Python object, including custom classes and objects
.
   - Joblib is more efficient than Pickle when working with large numpy arrays, making it a better choice for serializing large machine learning models
.

3. **Security Considerations**:
   - Both Pickle and Joblib use the pickle protocol under the hood, which can execute arbitrary code during unpickling. Therefore, it is important to only load models from trusted sources to avoid security risks
.

4. **Saving a Model with Pickle**:
   - To save a model using Pickle, you can use the `pickle.dump` function. This function serializes the model and writes it to a file. For example:
     ```python
     import pickle
     with open('model.pkl', 'wb') as file:
         pickle.dump(model, file)
     ```
   - This code opens a file in write-binary mode and uses Pickle to serialize the model into this file
.

5. **Loading a Model with Pickle**:
   - To load a model using Pickle, you can use the `pickle.load` function. This function reads the serialized model from a file and deserializes it. For example:
     ```python
     with open('model.pkl', 'rb') as file:
         model = pickle.load(file)
     ```
   - This code opens the file in read-binary mode and uses Pickle to deserialize the model from this file
.

6. **Saving a Model with Joblib**:
   - To save a model using Joblib, you can use the `joblib.dump` function. This function serializes the model and writes it to a file. For example:
     ```python
     from joblib import dump, load
     dump(model, 'model.joblib')
     ```
   - This code uses Joblib to serialize the model into a file named 'model.joblib'
.

7. **Loading a Model with Joblib**:
   - To load a model using Joblib, you can use the `joblib.load` function. This function reads the serialized model from a file and deserializes it. For example:
     ```python
     model = load('model.joblib')
     ```
   - This code uses Joblib to deserialize the model from the file named 'model.joblib'
.

8. **Performance Comparison**:
   - Joblib is generally faster than Pickle when dealing with large numpy arrays due to its optimizations for such data structures. However, Pickle may be faster for smaller models or models that do not contain large numpy arrays
.

9. **Use Cases**:
   - Use Pickle when you need to serialize small models or models that do not contain large numpy arrays. Use Joblib when you need to serialize large models or models that contain large numpy arrays
.

10. **Conclusion**:
    - Both Pickle and Joblib are powerful tools for model serialization. The choice between them depends on the size of your model and the data structures it contains. Always ensure that you load models from trusted sources to avoid security risks
.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)


,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [3]:
import pickle
version = '1.0.0'
filename_pkl = f'model_v{version}.pkl'
with open(filename_pkl, 'wb') as f:
    pickle.dump(pipeline, f)


In [4]:
import joblib
filename_joblib = f'model_v{version}.joblib'
joblib.dump(pipeline, filename_joblib)


['model_v1.0.0.joblib']

In [5]:
with open(filename_pkl, 'rb') as f:
    model_pickle = pickle.load(f)
preds_pickle = model_pickle.predict(X_test)


In [6]:
with open(filename_pkl, 'rb') as f:
    model_pickle = pickle.load(f)
preds_pickle = model_pickle.predict(X_test)


In [7]:
model_joblib = joblib.load(filename_joblib)
preds_joblib = model_joblib.predict(X_test)


In [8]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, preds_pickle))
print(accuracy_score(y_test, preds_joblib))


0.9649122807017544
0.9649122807017544


In [9]:
import os
versions = ['1.0.0', '1.1.0', '2.0.0']
for v in versions:
    fname = f'model_v{v}.joblib'
    joblib.dump(pipeline, fname)
os.listdir('.')


['best_tuned_model.pkl',
 'model_metadata.json',
 'model_v1.0.0.joblib',
 'model_v1.0.0.pkl',
 'model_v1.1.0.joblib',
 'model_v2.0.0.joblib',
 'svm_default_learning_curve.png',
 'svm_optimized_learning_curve.png',
 'Week_4_Day_3_Part1.ipynb',
 'Week_4_Day_3_Part2.ipynb']

In [10]:
import json
metadata = {
    'model_name': 'RandomForestPipeline',
    'versions': versions,
    'date_saved': {v: f'model_v{v}.joblib' for v in versions}
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f)


In [11]:
with open('model_metadata.json', 'r') as f:
    meta = json.load(f)
print(meta)


{'model_name': 'RandomForestPipeline', 'versions': ['1.0.0', '1.1.0', '2.0.0'], 'date_saved': {'1.0.0': 'model_v1.0.0.joblib', '1.1.0': 'model_v1.1.0.joblib', '2.0.0': 'model_v2.0.0.joblib'}}


Excercise

### Overview  
This exercise guides you through training a scikit-learn pipeline, saving it with both Pickle and Joblib, managing model versions and metadata, and exposing the model via a simple Flask API  ([9. Model persistence — scikit-learn 1.6.1 documentation](https://scikit-learn.org/stable/model_persistence.html?utm_source=chatgpt.com)).  

### Tasks  
1. **Data Preparation:** Load the Breast Cancer Wisconsin dataset (`sklearn.datasets.load_breast_cancer`), split into train/test (80/20), and standardize features  ([Save and Load Machine Learning Models in Python with scikit-learn](https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/?utm_source=chatgpt.com)).  
2. **Pipeline Construction:** Build an `sklearn.pipeline.Pipeline` with `StandardScaler` and `RandomForestClassifier(n_estimators=100, random_state=42)` and fit it on the training data  ([How to properly pickle sklearn pipeline when using custom ...](https://stackoverflow.com/questions/57888291/how-to-properly-pickle-sklearn-pipeline-when-using-custom-transformer?utm_source=chatgpt.com)).  
3. **Serialization:**  
   - Save the fitted pipeline using Pickle protocol 5 to `model_v1.pkl`.  
   - Save the same pipeline using Joblib to `model_v1.joblib`  ([Save Machine Learning Model Using Pickle and Joblib](https://www.analyticsvidhya.com/blog/2021/08/quick-hacks-to-save-machine-learning-model-using-pickle-and-joblib/?utm_source=chatgpt.com)).  
4. **Version Control:** Simulate versioning by retraining the pipeline with a different random seed, then saving as `model_v2.pkl`/`.joblib`; repeat for a third version. Create a `model_metadata.json` cataloging versions, file names, and timestamps  ([Machine Learning Model Serialization - Christopher Flynn](https://flynn.gg/blog/machine-learning-model-serialization/?utm_source=chatgpt.com)).  
5. **Deserialization & Validation:**  
   - Load each Pickle and Joblib file, run `.predict()` on the test set, and report accuracy.  
   - Measure and compare file sizes and load times between Pickle and Joblib  ([Pickle is over 10 times faster than joblib for save and load scikit ...](https://www.reddit.com/r/Python/comments/ypj13g/pickle_is_over_10_times_faster_than_joblib_for/?utm_source=chatgpt.com)).  
6. **Model Serving (Bonus):**  
   - Implement a minimal Flask app with two endpoints:  
     - `GET /models` returns available version metadata.  
     - `POST /predict` accepts JSON feature vectors and returns predicted classes using the latest model.  
   - Demonstrate sending requests via `requests` in Colab  ([Model Serialization using pickle and joblib - Kaggle](https://www.kaggle.com/code/tasnimniger/model-serialization-using-pickle-and-joblib?utm_source=chatgpt.com)).  

### Deliverables  
- A Colab notebook containing all code cells above.  
- The saved model files (`.pkl`, `.joblib`) and `model_metadata.json`.  
- A brief report (markdown cell) comparing Pickle vs Joblib in speed and size.  
- (Optional) The Flask app code cell demonstrating model serving.

In [12]:
# ==============================================================================
# TASK 1: Data Preparation
# ==============================================================================
import numpy as np
import pandas as pd
import time
import os
import json
import pickle
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
X, y = load_breast_cancer(return_X_y=True)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (455, 30), Test shape: (114, 30)


In [13]:
# ==============================================================================
# TASK 2: Pipeline Construction
# ==============================================================================
# Note: StandardScaler is included INSIDE the pipeline, so we don't need to
# manually scale X_train/X_test separately — the pipeline handles it during
# both .fit() and .predict()

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
print(f"Pipeline trained. Test accuracy: {pipeline.score(X_test, y_test):.4f}")

Pipeline trained. Test accuracy: 0.9649


In [14]:
# ==============================================================================
# TASK 3: Serialization (version 1)
# ==============================================================================
version = '1'

filename_pkl = f'model_v{version}.pkl'
with open(filename_pkl, 'wb') as f:
    pickle.dump(pipeline, f, protocol=5)   # Pickle protocol 5, as specified

filename_joblib = f'model_v{version}.joblib'
joblib.dump(pipeline, filename_joblib)

print(f"Saved {filename_pkl} and {filename_joblib}")

Saved model_v1.pkl and model_v1.joblib


In [16]:
# ==============================================================================
# TASK 4: Version Control — retrain with different seeds, save v2 and v3
# ==============================================================================
version_info = {}   # will collect metadata as we go

def train_and_save(version_str, random_seed):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=random_seed))
    ])
    pipe.fit(X_train, y_train)
    
    pkl_name = f'model_v{version_str}.pkl'
    joblib_name = f'model_v{version_str}.joblib'
    
    with open(pkl_name, 'wb') as f:
        pickle.dump(pipe, f, protocol=5)
    joblib.dump(pipe, joblib_name)
    
    acc = pipe.score(X_test, y_test)
    
    version_info[version_str] = {
        'pkl_file': pkl_name,
        'joblib_file': joblib_name,
        'random_state': random_seed,
        'test_accuracy': acc,
        'timestamp': pd.Timestamp.now().isoformat()
    }
    return pipe

# Version 1 already trained above — record its metadata too
version_info['1'] = {
    'pkl_file': filename_pkl,
    'joblib_file': filename_joblib,
    'random_state': 42,
    'test_accuracy': pipeline.score(X_test, y_test),
    'timestamp': pd.Timestamp.now().isoformat()
}

# Version 2 and 3 with different seeds (simulating retraining)
pipeline_v2 = train_and_save('2', random_seed=7)
pipeline_v3 = train_and_save('3', random_seed=99)

print("All versions trained and saved.")

# Create model_metadata.json cataloging versions, filenames, timestamps
metadata = {
    'model_name': 'RandomForestPipeline',
    'versions': list(version_info.keys()),
    'details': version_info
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved model_metadata.json")
with open('model_metadata.json', 'r') as f:
    print(json.load(f))

All versions trained and saved.
Saved model_metadata.json
{'model_name': 'RandomForestPipeline', 'versions': ['1', '2', '3'], 'details': {'1': {'pkl_file': 'model_v1.pkl', 'joblib_file': 'model_v1.joblib', 'random_state': 42, 'test_accuracy': 0.9649122807017544, 'timestamp': '2026-07-24T13:02:58.033403'}, '2': {'pkl_file': 'model_v2.pkl', 'joblib_file': 'model_v2.joblib', 'random_state': 7, 'test_accuracy': 0.9649122807017544, 'timestamp': '2026-07-24T13:02:58.283073'}, '3': {'pkl_file': 'model_v3.pkl', 'joblib_file': 'model_v3.joblib', 'random_state': 99, 'test_accuracy': 0.9649122807017544, 'timestamp': '2026-07-24T13:02:58.476579'}}}


In [17]:
# ==============================================================================
# TASK 5: Deserialization & Validation
# ==============================================================================
results = []

for v in ['1', '2', '3']:
    pkl_file = f'model_v{v}.pkl'
    joblib_file = f'model_v{v}.joblib'
    
    # --- Pickle: load + predict + time it ---
    start = time.time()
    with open(pkl_file, 'rb') as f:
        model_pkl = pickle.load(f)
    pkl_load_time = time.time() - start
    preds_pkl = model_pkl.predict(X_test)
    acc_pkl = accuracy_score(y_test, preds_pkl)
    
    # --- Joblib: load + predict + time it ---
    start = time.time()
    model_joblib = joblib.load(joblib_file)
    joblib_load_time = time.time() - start
    preds_joblib = model_joblib.predict(X_test)
    acc_joblib = accuracy_score(y_test, preds_joblib)
    
    # --- File sizes ---
    pkl_size_kb = os.path.getsize(pkl_file) / 1024
    joblib_size_kb = os.path.getsize(joblib_file) / 1024
    
    results.append({
        'Version': v,
        'Pickle Accuracy': acc_pkl,
        'Joblib Accuracy': acc_joblib,
        'Pickle Load Time (s)': pkl_load_time,
        'Joblib Load Time (s)': joblib_load_time,
        'Pickle Size (KB)': pkl_size_kb,
        'Joblib Size (KB)': joblib_size_kb
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

Version  Pickle Accuracy  Joblib Accuracy  Pickle Load Time (s)  Joblib Load Time (s)  Pickle Size (KB)  Joblib Size (KB)
      1         0.964912         0.964912              0.142364              0.034090         308.25293        318.783203
      2         0.964912         0.964912              0.018051              0.036112         316.53418        327.064453
      3         0.964912         0.964912              0.019042              0.034090         316.84668        327.376953


## Report: Pickle vs Joblib Comparison

Based on the results above (`comparison_df`):

- **Accuracy**: Identical for both — 0.9649 across all three versions, for both
  Pickle and Joblib. This is expected: both fully preserve the trained model's
  state, so predictions are unaffected by which library was used to save/load it.

- **File Size**: Joblib files were consistently slightly *larger* than Pickle
  in this case (e.g. v1: Pickle 308.25 KB vs Joblib 318.78 KB). This goes
  against the common assumption that Joblib is always smaller — for this
  model size, Joblib's overhead wasn't offset by compression gains.

- **Load Time**: Results were mixed. For v1, Pickle was slower to load
  (0.142s vs Joblib's 0.034s). For v2 and v3, both were fast and close
  (Pickle ~0.018–0.019s, Joblib ~0.034–0.036s) — Pickle was actually
  marginally faster on the smaller reloads, likely due to Joblib's extra
  overhead for numpy-array handling not paying off on a small model like this.

- **Takeaway**: For a model of this size (a single RandomForestClassifier
  with 100 trees on 30 features), the difference between Pickle and Joblib
  is minor and doesn't clearly favor one over the other. Joblib's advantages
  tend to show up more with much larger numpy-heavy models; for smaller
  models like this, plain Pickle is a perfectly reasonable choice.

In [18]:
# ==============================================================================
# TASK 6 (Bonus): Minimal Flask app for model serving
# ==============================================================================
# Note: this defines the Flask app code. Since Jupyter/Colab cells run
# synchronously, actually starting app.run() would block the notebook —
# so this is written to be run as a separate script (app.py) in practice.
# The comment block below shows how you'd run + test it locally.

flask_app_code = '''
from flask import Flask, request, jsonify
import joblib
import json
import numpy as np

app = Flask(__name__)

# Load metadata and the latest model version
with open('model_metadata.json', 'r') as f:
    metadata = json.load(f)

latest_version = metadata['versions'][-1]
latest_model = joblib.load(metadata['details'][latest_version]['joblib_file'])

@app.route('/models', methods=['GET'])
def get_models():
    return jsonify(metadata)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features = np.array(data['features']).reshape(1, -1)
    prediction = latest_model.predict(features)
    return jsonify({'prediction': int(prediction[0])})

if __name__ == '__main__':
    app.run(debug=True, port=5000)
'''

# Save the Flask app to a file
with open('app.py', 'w') as f:
    f.write(flask_app_code)

print("Flask app saved to app.py")
print("Run it locally with: python app.py")
print("Then test with requests, e.g.:")
print("""
import requests
sample = X_test[0].tolist()
response = requests.post('http://127.0.0.1:5000/predict', json={'features': sample})
print(response.json())
""")

Flask app saved to app.py
Run it locally with: python app.py
Then test with requests, e.g.:

import requests
sample = X_test[0].tolist()
response = requests.post('http://127.0.0.1:5000/predict', json={'features': sample})
print(response.json())



In [19]:
import requests

# Test GET /models — this one you CAN also test in browser, since it's a GET route
response = requests.get('http://127.0.0.1:5000/models')
print(response.json())

# Test POST /predict — must be done via requests, not browser
sample = X_test[0].tolist()
response = requests.post('http://127.0.0.1:5000/predict', json={'features': sample})
print(response.json())

{'details': {'1': {'joblib_file': 'model_v1.joblib', 'pkl_file': 'model_v1.pkl', 'random_state': 42, 'test_accuracy': 0.9649122807017544, 'timestamp': '2026-07-24T13:02:58.033403'}, '2': {'joblib_file': 'model_v2.joblib', 'pkl_file': 'model_v2.pkl', 'random_state': 7, 'test_accuracy': 0.9649122807017544, 'timestamp': '2026-07-24T13:02:58.283073'}, '3': {'joblib_file': 'model_v3.joblib', 'pkl_file': 'model_v3.pkl', 'random_state': 99, 'test_accuracy': 0.9649122807017544, 'timestamp': '2026-07-24T13:02:58.476579'}}, 'model_name': 'RandomForestPipeline', 'versions': ['1', '2', '3']}
{'prediction': 1}
